### 1. Basic Tasks

1. Install and authenticate the Databricks CLI using OAuth U2M against your workspace.

Installed the Databricks CLI locally and authenticated against the Databricks workspace using OAuth U2M.

Commands executed:

databricks -v

databricks auth login --host <workspace-url>

databricks auth profiles

databricks current-user me

Authentication was successfully verified using the current-user command.

2. Initialize a Declarative Automation Bundle project (databricks bundle init, or by hand) with one job
resource.

3. Run databricks bundle validate and databricks bundle deploy -t dev, then confirm the job appears in
your workspace.

1. Create the bundle using databricks CLI and select default python template 

`databrciks bundle init`

This creates a folder structure 
src/ : contains source code, motebooks and scripts.
resources/ : contains databricks resources such as jobs and pipelines 
databricks.yml file: main bundle configuration file for bundle

2. Create one job resource

create a job file in resource folder ingestion.job.yml

3. Validate bundle before deploying to check for configuration errors.

`databricks bundle validate -t dev`

This create or updates the resources defined in the bundle in databricks workspace 

4. Deploy after validation in the respectuve environment

`databricks bundle deploy -t dev`

now a new job will be created, run the job using command 

`databricks bundle run ingestion.job.yml -t dev`


### 2. Intermediate Tasks

4. Add a second target (staging or prod) to your databricks.yml with a different workspace host and run_as service principal, and deploy to it.

4. Add a second target (prod) to databricks.yml.

   Add another target in databricks.yml with a different workspace host
   and configure the Job to run as a service principal.

   Example:

   ```bash
   targets:
     dev:
       mode: development
       default: true
       workspace:
         host: https://<dev-workspace-url>

     prod:
       workspace:
         host: https://<prod-workspace-url>
       run_as:
         service_principal_name: <service-principal-application-id>
         ```

5. Configure M2M (service principal) authentication for the CLI and use it instead of your personal U2M login for a deploy command.

5. Configure M2M authentication using a Databricks service principal.

   First, create a service principal in Databricks and grant it the
   required permissions on the target workspace.

   Configure the Databricks CLI to authenticate using the service
   principal's OAuth credentials.

   Example environment variables:

   DATABRICKS_HOST=https://<workspace-url>
   DATABRICKS_CLIENT_ID=<service-principal-client-id>
   DATABRICKS_CLIENT_SECRET=<client-secret>

   The CLI can then authenticate using these credentials instead of
   my personal U2M login.

   Verify authentication:

   databricks current-user me

   Then deploy the bundle:

   databricks bundle deploy -t prod

6. Write a GitHub Actions workflow that runs databricks bundle validate on every pull request, without deploying anything.

```bash
name: Validate Databricks Bundle

on:
  pull_request:

jobs:
  validate:
    runs-on: ubuntu-latest

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate bundle
        run: databricks bundle validate
```

### 3. Advanced Tasks

7. Extend the GitHub Actions workflow to deploy to prod on merge to main using OIDC authentication (no stored secrets), including the correct permissions: id-token: write block.

```bash
name: Databricks CI/CD

on:
  pull_request:
  push:
    branches:
      - main

permissions:
  contents: read

jobs:
  validate:
    runs-on: ubuntu-latest

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate bundle
        run: databricks bundle validate -t prod

  deploy:
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    needs: validate
    runs-on: ubuntu-latest

    permissions:
      contents: read
      id-token: write

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Deploy to prod
        run: databricks bundle deploy -t prod
```

8. Design a rollback plan: if a bundle deploy to prod breaks a job, what CLI commands would you run to redeploy the previous working version quickly?

A practical rollback plan is to redeploy the exact Git commit that was last known to work, rather than trying to manually undo individual bundle resources.

Rollback commands

Assuming the last working commit is abc1234:

# 1. Check out the last known-good version
git checkout abc1234

# 2. Validate that bundle version
databricks bundle validate -t prod

# 3. Redeploy it to prod
databricks bundle deploy -t prod

If the repository is already checked out at the bad version, you can quickly move to the known-good commit:

git fetch origin
git checkout abc1234

databricks bundle validate -t prod
databricks bundle deploy -t prod
Recommended rollback procedure
Identify the last successful production commit from GitHub Actions.
Check out that exact commit.

Run:

databricks bundle validate -t prod

Redeploy:

databricks bundle deploy -t prod
Verify the affected job and its dependencies.
Once stable, revert/fix the bad change on main so the next deployment doesn't reintroduce it.

For a production setup, I'd also keep the Git commit SHA in the deployment logs. That makes the rollback target unambiguous and lets you reproduce the exact bundle configuration that was previously deployed.

9. Write a one-page onboarding guide for a new team member explaining how a change moves from a local databricks.yml edit to running safely in production, referencing the CLI, the bundle lifecycle, and the CI/CD workflow together.

Databricks Bundle CI/CD Onboarding Guide

When making a change, start by editing `databricks.yml` or the related bundle files locally. Create a feature branch and validate the bundle before pushing:

```bash
databricks bundle validate -t prod
```

If validation passes, commit your changes and open a pull request. GitHub Actions automatically runs the same validation on every PR. Nothing is deployed at this stage.

After the PR is reviewed and merged into `main`, GitHub Actions runs the production workflow. It validates the bundle again and then deploys it using:

```bash
databricks bundle deploy -t prod
```

The deployment uses GitHub OIDC authentication, so no Databricks password, PAT, or client secret is stored in GitHub. The deployment job uses the required `id-token: write` permission.

The overall flow is:

**Edit locally → Validate → Pull Request → CI validation → Merge to main → Deploy to prod**

If a production deployment causes a problem, find the last known-good Git commit, check it out, validate it, and redeploy:

```bash
git checkout <known-good-commit>
databricks bundle validate -t prod
databricks bundle deploy -t prod
```

This keeps production changes reviewed, validated, and traceable to a specific Git version.
